# 0. Preparamos el Entorno Spark-Hadoop

In [ ]:
# Revisar fuera de Google Colab, que exista la ruta https.
# Si no Existe, realizar la modificación de la ruta de acuerdo con la versión elegida:
#    - Normalmente cambia a partir de la versión de Spark.
#    - Revisar que el Nombre del Fichero Comprimido sea el asociado a la versión Hadoop 3.
#    - La descarga puede demorarse. Son alrededor de 370 Mb.
#    - La descarga se realiza en la carpeta /content del explorador de Google Colab.

!wget -q https://dlcdn.apache.org/spark/spark-3.4.4/spark-3.4.4-bin-hadoop3.tgz

In [ ]:
# Modificar el Nombre del fichero comprimido, para que coincida con el nombre de la celda anterior.
# Al descomprimirlo, se crea una carpeta con el mismo nombre.
!tar xf spark-3.4.4-bin-hadoop3.tgz

In [ ]:
# Descarga e Instalación de la libreria JDK
!pip install install-jdk

In [ ]:
# Instalación del JDK 8, donde se muestra la ruta en la cual es almacenado.
#    - Ejecutar una sola vez, de lo contrario aparecerá un error.
#    - Si aparece un error, descomentar la linea de desinstalación (jdk.uninstall('8')), y ejecutar la celda nuevamente.
# Esperar a que muestre la ruta.
import jdk
#jdk.uninstall('8')
jdk.install('8')

In [ ]:
# Instalación de la libreria de Pyspark
!pip install pyspark==3.4.4

### Definir Variables de Entorno

In [ ]:
# Definición de las Rutas de Entorno.
# Deben coincidir con las Instalaciones realizadas previamente.
import os
os.environ["SPARK_HOME"] = "/content/spark-3.4.4-bin-hadoop3"
os.environ["JAVA_HOME"] = "/root/.jdk/jdk8u432-b06"

In [ ]:
# Al terminar de Ejecutar esta Celda, se debería:
#  1.- Tener instalado el Spark Context
#  2.- Tener acceso a Spark UI
#  En caso de algún error, verificar los nombres de las rutas donde se hayan realizado las instalaciones previas.

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Test_spark").master("local[*]").getOrCreate()
spark

In [ ]:
# Crear Dataframe de Prueba para Verificar que estamos utilizando Pyspark
df = spark.createDataFrame([{"hello": "world"} for x in range(1004)])
print(df.count())
df.show(3, False)

# Comienzo del Tutorial con Pyspark
##### Este tutorial cubre los conceptos fundamentales de PySpark utilizando un caso práctico de análisis de datos de una tienda online.

## 1. Crear Dataframes con Datos

### 1.1 Crear los Archivos CSV de Ejemplo como Strings

In [ ]:
# ventas.csv
data_ventas = """
id_venta,fecha,id_cliente,id_producto,cantidad,precio_unitario
1,2024-01-01,101,1,2,29.99
2,2024-01-01,102,3,1,49.99
3,2024-01-02,103,2,3,19.99
4,2024-01-02,101,4,1,99.99
5,2024-01-03,104,1,1,29.99
6,2024-01-03,102,2,2,19.99
7,2024-01-04,105,5,1,149.99
8,2024-01-04,103,1,1,29.99
9,2024-01-05,101,3,2,49.99
10,2024-01-05,104,4,1,99.99
"""
# clientes.csv
data_clientes = """
id_cliente,nombre,email,ciudad,fecha_registro
101,Ana García,ana.garcia@email.com,Madrid,2023-01-15
102,Carlos Ruiz,carlos.ruiz@email.com,Barcelona,2023-02-20
103,María López,maria.lopez@email.com,Valencia,2023-03-10
104,Juan Martín,juan.martin@email.com,Sevilla,2023-04-05
105,Laura Torres,laura.torres@email.com,Bilbao,2023-05-22
"""
# productos.csv
data_productos = """
id_producto,nombre,categoria,stock,proveedor
1,Laptop Basic,Electrónica,50,TechCorp
2,Mouse Inalámbrico,Accesorios,200,AccessPro
3,Monitor 24",Electrónica,30,TechCorp
4,Tablet Pro,Electrónica,25,TechCorp
5,Laptop Premium,Electrónica,15,EliteTech
"""

In [ ]:
print(type(data_clientes_data))

### 1.2 Crear los Archivos CSV desde los Strings


In [ ]:
import os

def crear_archivo_csv(nombre, contenido):
    with open(nombre, 'w', encoding='utf-8') as f:
        f.write(contenido.strip())

crear_archivo_csv('data_ventas.csv', data_ventas)
crear_archivo_csv('data_clientes.csv', data_clientes)
crear_archivo_csv('data_productos.csv', data_productos)

### 1.3 Importar Librerias y Crear el Spark Context

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Crear SparkSession
spark = SparkSession.builder \
    .appName("TutorialPySpark") \
    .getOrCreate()

### 1.4 Definir Esquemas de Dataframes

In [ ]:
# Definir esquemas
ventas_schema = StructType([StructField("id_venta", IntegerType(), False),
                            StructField("fecha", DateType(), False),
                            StructField("id_cliente", IntegerType(), False),
                            StructField("id_producto", IntegerType(), False),
                            StructField("cantidad", IntegerType(), False),
                            StructField("precio_unitario", DoubleType(), False)
                            ])

clientes_schema = StructType([StructField("id_cliente", IntegerType(), False),
                              StructField("nombre", StringType(), False),
                              StructField("email", StringType(), False),
                              StructField("ciudad", StringType(), False),
                              StructField("fecha_registro", DateType(), False)
                              ])

productos_schema = StructType([StructField("id_producto", IntegerType(), False),
                               StructField("nombre", StringType(), False),
                               StructField("categoria", StringType(), False),
                               StructField("stock", IntegerType(), False),
                               StructField("proveedor", StringType(), False)
                               ])

### 1.5 Crear Dataframes desde CSV usando un Esquema

In [ ]:
# Leer DataFrames
df_ventas = spark.read.csv('data_ventas.csv', schema=ventas_schema, header=True)
df_clientes = spark.read.csv('data_clientes.csv', schema=clientes_schema, header=True)
df_productos = spark.read.csv('data_productos.csv', schema=productos_schema, header=True)

## 2. Operaciones Básicas



### 2.1. Mostrar Esquemas y Primeras Filas

In [ ]:
print("Esquema del Dataframe de Ventas:")
df_ventas.printSchema()
print("\nPrimeras Filas de Ventas:")
df_ventas.show(5)

### 2.2 Filtrado Básico

In [ ]:
# Ventas con cantidad mayor a 1
ventas_multiples = df_ventas.filter(col("cantidad") > 1)

# Productos de la categoría 'Electrónica'
productos_electronicos = df_productos.filter(col("categoria") == "Electrónica")

In [ ]:
ventas_multiples.show(5)
productos_electronicos.show(5)

### 2.3 Selección y Transformación de Columnas

In [ ]:
# Calcular Importe Total por Venta
ventas_con_total = df_ventas.withColumn("importe_total", col("cantidad") * col("precio_unitario"))
ventas_con_total.show(4)

## 3. Agregaciones y Agrupamientos

### 3.1 Ventas Totales por Día

In [ ]:
ventas_por_dia = df_ventas.groupBy("fecha") \
                          .agg(sum(col("cantidad") * col("precio_unitario")).alias("venta_total"),
                               count("id_venta").alias("num_ventas")
                              )
ventas_por_dia.show(4)

### 3.2 Productos más Vendidos

In [ ]:
productos_mas_vendidos = df_ventas.groupBy("id_producto") \
    .agg(
       sum("cantidad").alias("unidades_vendidas"),
        sum(col("cantidad") * col("precio_unitario")).alias("importe_total")
    ) \
    .orderBy(desc("unidades_vendidas"))
productos_mas_vendidos.show(3)

### 3.3 Ventas por Ciudad del Cliente

In [ ]:
df_ventas.printSchema()

In [ ]:
ventas_por_ciudad = df_ventas.join(df_clientes, "id_cliente") \
    .groupBy("ciudad") \
    .agg(
        sum(col("cantidad") * col("precio_unitario")).alias("venta_total"),
        countDistinct("id_cliente").alias("num_clientes")
    ) \
    .orderBy(desc("venta_total"))
    ventas_por_ciudad.show()

## 4. Joins y Análisis Complejos

### 4.1 Análisis Detallado de Ventas

In [ ]:
df_ventas.printSchema()
df_clientes.printSchema()
df_productos.printSchema()

In [ ]:
analisis_ventas = df_ventas \
    .join(df_clientes, "id_cliente") \
    .join(df_productos, "id_producto") \
    .select("id_venta", "fecha", "nombre", "ciudad",
            "productos.nombre".alias("producto"),
            "cantidad", "precio_unitario",
            (col("cantidad") * col("precio_unitario")).alias("importe_total")
            )
analisis_ventas.show(5)

### 4.2 Análisis de Productos por Proveedor

In [ ]:
analisis_proveedor = df_productos \
    .join(df_ventas, "id_producto") \
    .groupBy("proveedor", "categoria") \
    .agg(
        sum("cantidad").alias("unidades_vendidas"),
        sum(col("cantidad") * col("precio_unitario")).alias("importe_total"),
        avg("precio_unitario").alias("precio_promedio")
    )
analisis_proveedor.show(5)

### 4.3 Clientes más Valiosos

In [ ]:
clientes_valiosos = df_ventas \
    .join(df_clientes, "id_cliente") \
    .groupBy("id_cliente", "nombre", "ciudad") \
    .agg(
        sum(col("cantidad") * col("precio_unitario")).alias("total_compras"),
        count("id_venta").alias("num_compras"),
        avg(col("cantidad") * col("precio_unitario")).alias("ticket_promedio")
    ) \
    .orderBy(desc("total_compras"))
clientes_valiosos.show(5)

"""
EJERCICIOS PRÁCTICOS SUGERIDOS:

1. Calcular el porcentaje de ventas por categoría de producto
2. Encontrar los días con mayores y menores ventas
3. Analizar la frecuencia de compra de los clientes
4. Calcular el stock restante de productos después de las ventas
5. Identificar productos que necesitan reposición (stock < 20)
"""

# Ejemplo de solución para el ejercicio 1
ventas_por_categoria = df_ventas \
    .join(df_productos, "id_producto") \
    .groupBy("categoria") \
    .agg(sum(col("cantidad") * col("precio_unitario")).alias("venta_total")) \
    .withColumn("porcentaje",
        col("venta_total") / sum("venta_total").over() * 100)

print("\nVentas por categoría:")
ventas_por_categoria.show()